# Amparo · demo

Una persona sube la foto del documento que le llegó, escribe su pregunta y recibe:

1. **la respuesta**: las cinco líneas de Amparo;
2. **el por qué**: la regla del art. 6 del Decreto 2591 que decidió, los hechos que la
   activaron y las normas que consultó el agente;
3. **lo que leímos de su documento**, rotulado como lectura de un modelo **no verificada**,
   con un aviso cuando el texto trae un plazo que no se pudo leer como dato.

No es otro sistema: importa `m4/pipeline/multimodal.py`, el mismo código que evalúa
`M4_pipeline.ipynb`. Colab con GPU **T4** y el secreto `GROQ_API_KEY`. Al final aparece un
enlace público (`*.gradio.live`) que dura mientras la sesión esté abierta.

> **Plan B.** Si en la demo falla la red, Groq o la GPU, la salida grabada de m01, m06 y m10
> está en `m4/corridas/<id>/plan_b_demo.md`, rotulada como grabación.

In [ ]:
%pip install -q "transformers>=4.45" accelerate pillow qwen-vl-utils pandas groq sentence-transformers gradio
print('Listo.')

In [ ]:
# El mismo clon que el notebook de evaluacion: el tag fijo, no main.
import os, subprocess, sys
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'm4-demo'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'm4' / 'harness').is_dir()), None)
if RAIZ is None:
    if not Path('Amparo').is_dir():
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
os.chdir(RAIZ)
sys.path[:0] = [str(RAIZ / 'm4' / 'pipeline'), str(RAIZ / 'm4' / 'harness')]
_git = lambda *a: subprocess.run(['git', *a], capture_output=True, text=True).stdout.strip()
print('version', _git('describe', '--tags', '--always'), '· commit', _git('rev-parse', '--short', 'HEAD'))

In [ ]:
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('clave de Groq cargada desde los secretos de Colab')
except Exception as ex:
    if not os.environ.get('GROQ_API_KEY'):
        raise SystemExit('Falta GROQ_API_KEY: icono de la llave -> GROQ_API_KEY, con acceso '
                         'para este notebook. (' + type(ex).__name__ + ')')

In [ ]:
# Carga el lector visual y enchufa M3: tarda unos minutos la primera vez.
from multimodal import Amparo
amparo = Amparo()
print('Amparo listo')

In [ ]:
import gradio as gr
from PIL import Image

IMG = RAIZ / 'm4' / 'eval_set' / 'imagenes'
EJEMPLOS = [
    ['Me llegó esta respuesta de mi EPS. Tengo 68 años y el ortopedista me ordenó esa cirugía '
     'porque ya casi no puedo caminar. ¿Qué mecanismo constitucional me corresponde?',
     str(IMG / 'doc01_respuesta_eps.png')],
    ['Somos cuatro veredas que tomamos agua de esa quebrada y la curtiembre sigue vertiendo. '
     'Esta es la respuesta que nos dio la corporación ambiental. ¿Qué mecanismo constitucional '
     'nos corresponde?', str(IMG / 'doc04_respuesta_car.png')],
    ['Es una foto de una notificación que me dejaron debajo de la puerta y se mojó. ¿De qué '
     'fecha es y en cuántos días se me vence el plazo?', str(IMG / 'doc06_notificacion_borrosa.jpg')],
]


def responder(pregunta, imagen):
    if not (pregunta or '').strip():
        return '_Escribe tu pregunta._'
    return amparo.responder_markdown(pregunta.strip(), imagen)


demo = gr.Interface(
    fn=responder,
    inputs=[gr.Textbox(label='Tu pregunta', lines=4),
            gr.Image(type='pil', label='Foto del documento (opcional)')],
    outputs=gr.Markdown(label='Amparo'),
    examples=EJEMPLOS,
    title='Amparo · ¿qué acción constitucional me corresponde?',
    description=('Sube la foto del documento que te llegó y escribe tu pregunta. Amparo lee el '
                 'documento, aplica las reglas del art. 6 del Decreto 2591 y te dice por qué. '
                 '**No sustituye la asesoría de un abogado.** Los documentos de ejemplo son '
                 'sintéticos.'),
)
demo.launch(share=True)